# ProphetGP 사용 예시

이 노트북은 ProphetGP의 핵심 기능(학습, 후보 추천, 데이터셋 추가)을 빠르게 실행해보는 예시입니다.

In [ ]:
# 필요 시 주석 해제 후 설치
# !pip install -e .[dev]

In [1]:
from pathlib import Path

from prophet_gp.config import load_config
from prophet_gp.pipeline.trainer import ProphetGPPipeline
from prophet_gp.data.dataset import ReactionDatasetService

# 노트북 실행 위치가 notebooks/여도 안전하게 프로젝트 루트를 찾는다.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "sample_emission.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "sample" / "sample_data.csv"
NEW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "new_batch.csv"
MERGED_OUT_PATH = PROJECT_ROOT / "data" / "raw" / "reactions_merged.csv"

config = load_config(CONFIG_PATH)
pipeline = ProphetGPPipeline(config)
dataset_service = ReactionDatasetService(config.data)

print("Project root:", PROJECT_ROOT)
print("Config loaded:", config.model_dump())

Project root: c:\Projects\ProphetGP
Config loaded: {'data': {'reactant_column': 'Mol.1', 'target_column': 'Emission Peak', 'reactant_delimiter': '|', 'ignore_columns': ['FWHM', 'PLQY'], 'explicit_condition_types': {'Temperature': 'continuous'}}, 'featurization': {'featuriser': 'ecfp_fingerprints', 'combine_strategy': 'concat'}, 'optimization': {'objective': 'target', 'target_value': 495.0, 'n_restarts': 10, 'raw_samples': 128, 'target_search_size': 5000, 'n_candidates': 3}}


In [2]:
# 1) 사용 가능한 featuriser 확인
available_featurisers = pipeline.featurizers.available()
print("Available featurisers count:", len(available_featurisers))
print(available_featurisers[:20])  # 앞쪽 일부만 출력

Available featurisers count: 9
['bag_of_characters', 'drfp', 'ecfp_fingerprints', 'fragments', 'molecular_graphs', 'mqn_features', 'one_hot', 'rdkit_descriptors', 'rxnfp']


In [ ]:
# 2) 학습
artifacts = pipeline.train_from_csv(DATA_PATH)
print("Train rows:", artifacts.x_train.shape[0])
print("Feature dims:", artifacts.x_train.shape[1])

c:\Users\sung1234\AppData\Local\Programs\Python\Python39\lib\site-packages\botorch\models\utils\assorted.py:174: InputDataWarning: Input data is not contained to the unit cube. Please consider min-max scaling the input data.
  warnings.warn(msg, InputDataWarning)
c:\Users\sung1234\AppData\Local\Programs\Python\Python39\lib\site-packages\botorch\models\utils\assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([506.4375], dtype=torch.float64), std = tensor([43.6715], dtype=torch.float64)). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)


Train rows: 16
Feature dims: 2049


In [6]:
# 3) 다음 실험 조건 후보 추천 (raw + 해석 결과)
# strategy: "best_output" | "best_information"
n_candidates = 5
strategy = "best_output"
suggestions = pipeline.suggest_next_experiments(
    artifacts,
    n_candidates=n_candidates,
    strategy=strategy,
)

print("Strategy:", strategy)
print("Raw candidates shape:", suggestions.raw_candidates.shape)
print("Decoded candidates:")
for idx, row in enumerate(suggestions.decoded_candidates, 1):
    print(
        f"- candidate_{idx}: mean={row['predicted_target_mean']:.3f}, "
        f"std={row['predicted_target_std']:.3f}, gap={row['target_gap']}"
    )
    print("  mapped input:", row)

suggestions.raw_candidates

Candidates shape: (5, 2049)


array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.95617388],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -1.09455309],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -1.03135187],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -0.76131675],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        , -1.12122201]])

In [ ]:
# 4) 신규 배치 데이터 append
# 파일이 준비되어 있지 않으면 이 셀은 건너뛰세요.
merged = dataset_service.append_csv(DATA_PATH, NEW_DATA_PATH, MERGED_OUT_PATH)
print("Merged rows:", len(merged))
print("Saved to:", MERGED_OUT_PATH)

## 입력 데이터 포맷 가이드

- `reactants`: 반응물 리스트 (`|` 구분)
- `target`: 예측/최적화 대상 물성값
- 그 외 컬럼: 반응 조건(문자열/숫자 모두 가능, 자동 타입 추론 + config override 지원)